In [4]:
import pandas as pd
import ccxt
import plotly
print("ccxt:", ccxt.__version__)
print("pandas:", pd.__version__)
print("plotly:", plotly.__version__)


ccxt: 4.5.88
pandas: 3.0.6
plotly: 7.1.0


In [6]:
exchange = ccxt.binance()

print("Rate limiting enabled:", exchange.enableRateLimit)
print("Minimum delay between requests (ms):", exchange.rateLimit)

Rate limiting enabled: True
Minimum delay between requests (ms): 50


In [23]:
candles = exchange.fetch_ohlcv("BTC/USDT", timeframe="1d", limit=5)

print(type(candles), len(candles))
for candle in candles:
    print(candle)

<class 'list'> 5
[1791763200000, 83773.12, 84352.0, 81750.09, 81850.0, 13529.90112]
[1791849600000, 81850.01, 84168.0, 81033.66, 83169.0, 14784.0627]
[1791936000000, 83169.01, 84146.72, 81444.42, 81682.01, 18307.57283]
[1792022400000, 81682.02, 82634.3, 79800.0, 82245.93, 22563.14627]
[1792108800000, 82245.93, 82940.81, 80809.25, 81492.01, 11268.82852]


In [17]:
for i in range(len(candles)-1) :
    date=pd.to_datetime(candles[i][0], unit="ms", utc=True)
    print(date, "close:", candles[i][4], "next open:", candles[i+1][1])
    print("------------")
print(pd.to_datetime(candles[-1][0], unit="ms", utc=True), "close:", candles[-1][4])

2026-10-12 00:00:00+00:00 close: 81850.0 next open: 81850.01
------------
2026-10-13 00:00:00+00:00 close: 83169.0 next open: 83169.01
------------
2026-10-14 00:00:00+00:00 close: 81682.01 next open: 81682.02
------------
2026-10-15 00:00:00+00:00 close: 82245.93 next open: 82245.93
------------
2026-10-16 00:00:00+00:00 close: 80878.2


In [24]:
all_candles = []
since = exchange.parse8601("2017-01-01T00:00:00Z")

while True:
    batch = exchange.fetch_ohlcv("BTC/USDT", timeframe="1d", since=since, limit=1000)
    if len(batch) == 0:
        break
    all_candles.extend(batch)

    first_date = pd.to_datetime(batch[0][0], unit="ms", utc=True)
    last_date = pd.to_datetime(batch[-1][0], unit="ms", utc=True)
    print(f"Fetched {len(batch)} candles: {first_date.date()} -> {last_date.date()}")

    if len(batch) < 1000:
        break
    since = batch[-1][0] + 1

print("Total candles:", len(all_candles))

Fetched 1000 candles: 2017-08-17 -> 2020-05-12
Fetched 1000 candles: 2020-05-13 -> 2023-02-06
Fetched 1000 candles: 2023-02-07 -> 2025-11-02
Fetched 348 candles: 2025-11-03 -> 2026-10-16
Total candles: 3348


In [26]:
## Step 4 — Build the DataFrame
columns=["timestamp", "open", "high", "low", "close", "volume"]
df=pd.DataFrame(all_candles, columns=columns)

df["date"]=pd.to_datetime(df["timestamp"], unit="ms", utc=True)
df=df.set_index("date")

In [40]:
df.tail()

,timestamp,open,high,low,close,volume
date,,,,,,
2026-10-12 00:00:00+00:00,1791763200000,83773.12,84352.00,81750.09,81850.00,13529.90112
2026-10-13 00:00:00+00:00,1791849600000,81850.01,84168.00,81033.66,83169.00,14784.06270
2026-10-14 00:00:00+00:00,1791936000000,83169.01,84146.72,81444.42,81682.01,18307.57283
2026-10-15 00:00:00+00:00,1792022400000,81682.02,82634.30,79800.00,82245.93,22563.14627
2026-10-16 00:00:00+00:00,1792108800000,82245.93,82940.81,80809.25,81275.86,11831.66395
